In [ ]:
import pandas as pd
import ijson
from itertools import islice
import json 
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()
# I dare you to try to get my API keys
OpenAI_api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OpenAI_api_key)

In [ ]:
#From Kaggle
fileName = 'data/arxiv-metadata-oai-snapshot.json'
num_records_to_read = 10_000
data = []
def clean_authors(raw_author):
    authors = []
    for sublist in raw_author:
        authors.append((sublist[0].strip() + ' ' + sublist[1].strip() + ' ' + sublist[2].strip()).strip())
    return authors
with open(fileName, 'r') as f:
        for line in islice(f, num_records_to_read):
            raw_line = json.loads(line)
            extracted_info = {'id': raw_line['id'], 'title': raw_line['title'], 'abstract': raw_line['abstract'], 'authors': clean_authors(raw_line['authors_parsed']), 'categories': raw_line['categories'], 'update_date': raw_line['update_date']}
            data.append(extracted_info)
df_raw = pd.DataFrame(data)
df_raw['categories'] = df_raw['categories'].apply(lambda x: x.split(' ')[0])
df_raw['categories'] = df_raw['categories'].apply(lambda x: x.split('.')[0])
df_raw['title_abstract'] = df_raw['title'] + '\n' + df_raw['abstract']

In [136]:
df_raw

,id,title,abstract,authors,categories,update_date,title_abstract
0,0704.0001,Calculation of prompt diphoton production cros...,A fully differential calculation in perturba...,"[Balázs C., Berger E. L., Nadolsky P. M., Yuan...",hep-ph,2008-11-26,Calculation of prompt diphoton production cros...
1,0704.0002,Sparsity-certifying Graph Decompositions,"We describe a new algorithm, the $(k,\ell)$-...","[Streinu Ileana, Theran Louis]",math,2008-12-13,Sparsity-certifying Graph Decompositions\n We...
2,0704.0003,The evolution of the Earth-Moon system based o...,The evolution of Earth-Moon system is descri...,[Pan Hongjun],physics,2008-01-13,The evolution of the Earth-Moon system based o...
3,0704.0004,A determinant of Stirling cycle numbers counts...,We show that a determinant of Stirling cycle...,[Callan David],math,2007-05-23,A determinant of Stirling cycle numbers counts...
4,0704.0005,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,In this paper we show how to compute the $\L...,"[Abu-Shammala Wael, Torchinsky Alberto]",math,2013-10-15,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...
...,...,...,...,...,...,...,...
9995,0706.1309,Superconducting behavior of the solid solution...,The compound Y2PdGe3 was earlier reported by...,"[Iyer Kartik K, Sampathkumaran E. V.]",cond-mat,2009-11-13,Superconducting behavior of the solid solution...
9996,0706.1310,The Geometer's Toolkit to String Compactificat...,These lecture notes are meant to serve as an...,[Reffert S.],hep-th,2009-09-29,The Geometer's Toolkit to String Compactificat...
9997,0706.1311,Compactified moduli of projective bundles,We present a method for compactifying stacks...,[Lieblich Max],math,2018-06-18,Compactified moduli of projective bundles\n W...
9998,0706.1312,Entrelacement d'alg\`ebres de Lie [Wreath prod...,Full details are given for the definition an...,"[Coffi-Nketsia Barben-Jean, Haddad Labib]",math,2007-06-12,Entrelacement d'alg\`ebres de Lie [Wreath prod...


In [ ]:
import concurrent.futures
import time
import logging 
from openai import RateLimitError 

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def get_embedding(text, model="text-embedding-3-large", retries=5, initial_delay=1):
    text = text.replace("\n", " ")
    delay = initial_delay
    for attempt in range(retries):
        try:
            return client.embeddings.create(input = text,
                                            model=model,
                                            encoding_format='float',
                                            dimensions=256).data[0].embedding
        except RateLimitError as e:
            wait_time = delay * (2 ** attempt)
            logging.warning(f"Rate limit hit (attempt {attempt + 1}/{retries}): {e}. Retrying in {wait_time:.2f}s...")
            time.sleep(wait_time)
        except Exception as e:
            wait_time = delay * (2 ** attempt)
            logging.warning(f"API call failed (attempt {attempt + 1}/{retries}): {e}. Retrying in {wait_time:.2f}s...")
            time.sleep(wait_time)

    logging.error(f"API call failed after {retries} attempts for text snippet: {text[:100]}...")
    return None 

MAX_WORKERS = 20 
texts_to_embed = df_raw['title_abstract'].tolist()
embeddings = [None] * len(texts_to_embed)

start_time = time.time()


with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_to_index = {executor.submit(get_embedding, text, initial_delay=1): i for i, text in enumerate(texts_to_embed)}

    processed_count = 0
    for future in concurrent.futures.as_completed(future_to_index):
        index = future_to_index[future]
        try:
            result = future.result()
            embeddings[index] = result
            processed_count += 1
            if processed_count % 100 == 0:
                 current_time = time.time()
                 elapsed_time = current_time - start_time
                 rate = processed_count / elapsed_time if elapsed_time > 0 else 0
                 logging.info(f"Generated {processed_count}/{len(texts_to_embed)} embeddings... (Rate: {rate:.2f} embeddings/sec)")
        except Exception as exc:
            logging.error(f'Text at index {index} generated an exception during future processing: {exc}')

end_time = time.time()
total_time = end_time - start_time
final_rate = len(texts_to_embed) / total_time if total_time > 0 else 0
logging.info(f"Embedding generation finished. Processed {processed_count} texts in {total_time:.2f} seconds (Average rate: {final_rate:.2f} embeddings/sec).")


df_raw['embedding'] = embeddings

failed_count = df_raw['embedding'].isnull().sum()
if failed_count > 0:
    logging.warning(f"{failed_count} embeddings could not be generated after retries.")
df_embedded_cleaned = df_raw.dropna(subset=['embedding']).copy()


2025-05-01 17:19:10,377 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-05-01 17:19:10,422 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-05-01 17:19:10,433 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-05-01 17:19:10,436 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-05-01 17:19:10,463 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-05-01 17:19:10,493 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-05-01 17:19:10,499 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-05-01 17:19:10,507 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-05-01 17:19:10,512 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-05-01 17:19:10,516 - INFO - HTTP

In [138]:
df_embedded_cleaned


,id,title,abstract,authors,categories,update_date,title_abstract,embedding
0,0704.0001,Calculation of prompt diphoton production cros...,A fully differential calculation in perturba...,"[Balázs C., Berger E. L., Nadolsky P. M., Yuan...",hep-ph,2008-11-26,Calculation of prompt diphoton production cros...,"[-0.031572502, -0.071272574, -0.03544873, 0.12..."
1,0704.0002,Sparsity-certifying Graph Decompositions,"We describe a new algorithm, the $(k,\ell)$-...","[Streinu Ileana, Theran Louis]",math,2008-12-13,Sparsity-certifying Graph Decompositions\n We...,"[0.021066308, 0.038581938, -0.08546993, -0.009..."
2,0704.0003,The evolution of the Earth-Moon system based o...,The evolution of Earth-Moon system is descri...,[Pan Hongjun],physics,2008-01-13,The evolution of the Earth-Moon system based o...,"[-0.05145531, -0.09710397, -0.032968476, 0.018..."
3,0704.0004,A determinant of Stirling cycle numbers counts...,We show that a determinant of Stirling cycle...,[Callan David],math,2007-05-23,A determinant of Stirling cycle numbers counts...,"[0.10047025, -0.0012192992, -0.053911783, 0.00..."
4,0704.0005,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,In this paper we show how to compute the $\L...,"[Abu-Shammala Wael, Torchinsky Alberto]",math,2013-10-15,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,"[-0.05475762, -0.0073111886, -0.05876853, -0.0..."
...,...,...,...,...,...,...,...,...
9995,0706.1309,Superconducting behavior of the solid solution...,The compound Y2PdGe3 was earlier reported by...,"[Iyer Kartik K, Sampathkumaran E. V.]",cond-mat,2009-11-13,Superconducting behavior of the solid solution...,"[0.039241053, 0.0025156087, -0.043113098, -0.0..."
9996,0706.1310,The Geometer's Toolkit to String Compactificat...,These lecture notes are meant to serve as an...,[Reffert S.],hep-th,2009-09-29,The Geometer's Toolkit to String Compactificat...,"[0.043007173, 0.043792848, -0.053630866, -0.05..."
9997,0706.1311,Compactified moduli of projective bundles,We present a method for compactifying stacks...,[Lieblich Max],math,2018-06-18,Compactified moduli of projective bundles\n W...,"[0.02549525, -0.037114616, -0.08850927, 0.0105..."
9998,0706.1312,Entrelacement d'alg\`ebres de Lie [Wreath prod...,Full details are given for the definition an...,"[Coffi-Nketsia Barben-Jean, Haddad Labib]",math,2007-06-12,Entrelacement d'alg\`ebres de Lie [Wreath prod...,"[0.016954454, 0.0037162192, -0.053173788, -0.0..."


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
import numpy as np

embeddings = np.array(df_embedded_cleaned['embedding'].tolist(), dtype=np.float32)

print("Normalizing embeddings...")
embeddings_normalized = normalize(embeddings, norm='l2', axis=1)
print("Normalization complete.")

df_embedded_cleaned_clustred = df_embedded_cleaned.copy()

min_clusters = 3
max_clusters = 15
cluster_range = range(min_clusters, max_clusters + 1)

print(f"Calculating K-Means clusters for k={min_clusters} to k={max_clusters}...")

for k in cluster_range:
    print(f"  Fitting K-Means with {k} clusters...")
    kmeans = KMeans(n_clusters=k,
                    random_state=42,
                    n_init='auto')

    cluster_labels = kmeans.fit_predict(embeddings_normalized)

    column_name = f'kmeans_k{k}'
    df_embedded_cleaned_clustred[column_name] = cluster_labels
    print(f"  Stored results in column '{column_name}'.")

print("K-Means clustering calculations complete for all k values.")


/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning:

divide by zero encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning:

overflow encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning:

invalid value encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/opt/anaconda3/envs/arXiv/li

Normalizing embeddings...
Normalization complete.
Calculating K-Means clusters for k=3 to k=15...
  Fitting K-Means with 3 clusters...
  Stored results in column 'kmeans_k3'.
  Fitting K-Means with 4 clusters...
  Stored results in column 'kmeans_k4'.
  Fitting K-Means with 5 clusters...
  Stored results in column 'kmeans_k5'.
  Fitting K-Means with 6 clusters...
  Stored results in column 'kmeans_k6'.
  Fitting K-Means with 7 clusters...
  Stored results in column 'kmeans_k7'.
  Fitting K-Means with 8 clusters...


/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning:

divide by zero encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning:

overflow encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning:

invalid value encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/opt/anaconda3/envs/arXiv/li

  Stored results in column 'kmeans_k8'.
  Fitting K-Means with 9 clusters...
  Stored results in column 'kmeans_k9'.
  Fitting K-Means with 10 clusters...
  Stored results in column 'kmeans_k10'.
  Fitting K-Means with 11 clusters...


/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning:

divide by zero encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning:

overflow encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning:

invalid value encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/opt/anaconda3/envs/arXiv/li

  Stored results in column 'kmeans_k11'.
  Fitting K-Means with 12 clusters...
  Stored results in column 'kmeans_k12'.
  Fitting K-Means with 13 clusters...
  Stored results in column 'kmeans_k13'.
  Fitting K-Means with 14 clusters...


/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning:

divide by zero encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning:

overflow encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning:

invalid value encountered in matmul



  Stored results in column 'kmeans_k14'.
  Fitting K-Means with 15 clusters...
  Stored results in column 'kmeans_k15'.
K-Means clustering calculations complete for all k values.


/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning:

divide by zero encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning:

overflow encountered in matmul

/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning:

invalid value encountered in matmul



In [ ]:
import umap
import numpy as np

embeddings_array = np.array(df_embedded_cleaned_clustred['embedding'].tolist(), dtype=np.float32)

print("Starting UMAP reduction to 3 dimensions...")
reducer_3d = umap.UMAP(n_components=3,
                       n_neighbors=15,
                       min_dist=0.1,
                       metric='cosine',
                       random_state=42)

embedding_3d = reducer_3d.fit_transform(embeddings_array)
print("UMAP reduction complete.")

df_plot = df_embedded_cleaned_clustred.copy()
df_plot['umap_x'] = embedding_3d[:, 0]
df_plot['umap_y'] = embedding_3d[:, 1]
df_plot['umap_z'] = embedding_3d[:, 2]
print("UMAP coordinates added to DataFrame 'df_plot'.")

df_plot.head()


/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.



Starting UMAP reduction to 3 dimensions...


/opt/anaconda3/envs/arXiv/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



UMAP reduction complete.
UMAP coordinates added to DataFrame 'df_plot'.


,id,title,abstract,authors,categories,update_date,title_abstract,embedding,kmeans_k3,kmeans_k4,...,kmeans_k9,kmeans_k10,kmeans_k11,kmeans_k12,kmeans_k13,kmeans_k14,kmeans_k15,umap_x,umap_y,umap_z
0,0704.0001,Calculation of prompt diphoton production cros...,A fully differential calculation in perturba...,"[Balázs C., Berger E. L., Nadolsky P. M., Yuan...",hep-ph,2008-11-26,Calculation of prompt diphoton production cros...,"[-0.031572502, -0.071272574, -0.03544873, 0.12...",1,2,...,7,7,7,7,1,1,1,4.635396,8.555543,7.953991
1,0704.0002,Sparsity-certifying Graph Decompositions,"We describe a new algorithm, the $(k,\ell)$-...","[Streinu Ileana, Theran Louis]",math,2008-12-13,Sparsity-certifying Graph Decompositions\n We...,"[0.021066308, 0.038581938, -0.08546993, -0.009...",0,3,...,8,9,9,9,9,9,9,2.491081,2.930961,7.375481
2,0704.0003,The evolution of the Earth-Moon system based o...,The evolution of Earth-Moon system is descri...,[Pan Hongjun],physics,2008-01-13,The evolution of the Earth-Moon system based o...,"[-0.05145531, -0.09710397, -0.032968476, 0.018...",2,1,...,5,4,4,5,7,5,5,6.578194,6.472511,7.252895
3,0704.0004,A determinant of Stirling cycle numbers counts...,We show that a determinant of Stirling cycle...,[Callan David],math,2007-05-23,A determinant of Stirling cycle numbers counts...,"[0.10047025, -0.0012192992, -0.053911783, 0.00...",0,3,...,8,9,9,9,9,9,9,2.193058,3.434114,7.970769
4,0704.0005,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,In this paper we show how to compute the $\L...,"[Abu-Shammala Wael, Torchinsky Alberto]",math,2013-10-15,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,"[-0.05475762, -0.0073111886, -0.05876853, -0.0...",0,3,...,8,9,8,8,8,8,8,3.129913,3.533654,7.135292


In [ ]:
df_plot.drop(columns=['embedding', 'abstract', 'title_abstract']).to_json('data/data.json', orient='records')